In [ ]:
!pip install opencv-contrib-python

In [ ]:
!pip install opencv-python

In [2]:
!pip install ultralytics

  Obtaining dependency information for ultralytics from https://files.pythonhosted.org/packages/9e/20/fcef1ebb10c8eb739d8ae8c3ead86c89d78c0121c23c5df5b85671ef3ef6/ultralytics-8.3.23-py3-none-any.whl.metadata
  Obtaining dependency information for matplotlib>=3.3.0 from https://files.pythonhosted.org/packages/8b/ce/15b0bb2fb29b3d46211d8ca740b96b5232499fc49200b58b8d571292c9a6/matplotlib-3.9.2-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pillow>=7.1.2 from https://files.pythonhosted.org/packages/dc/83/1470c220a4ff06cd75fc609068f6605e567ea51df70557555c2ab6516b2c/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for scipy>=1.4.1 from https://files.pythonhosted.org/packages/ea/c2/5ecadc5fcccefaece775feadcd795060adf5c3b29a883bff0e678cfe89af/scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
 

ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
    unknown package:
        Expected sha256 716e389b694c4bb564b4fc0c51bc84d381735e0d39d3f26ec1af2556ec6aad94
             Got        289d9a7541a18231d1aaf40a660841bf6c2b86f9218b23182fe1c2f8bd1f90d1


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
import numpy
import crop_image
from ball_detection import isOpen
import os

print("done")

done


в этой ячейке код для создания заднего фона путем ручного выбора кадров и кусков изображения свободных от частей, которые будут двигаться

In [2]:
#creating background simple version
def catch_frame(to_file: str, from_file: str = "../materials_part1/5.mkv", hdiv: float = 1, wdiv: float = 1,
                image_id: int = 0, quater_id: int = 0) -> numpy.ndarray:
    cam = cv2.VideoCapture(from_file)
    cv2.namedWindow("background take")
    isclosed = False
    pframe = None
    while not isclosed:
        isclosed |= not isOpen("background take")
        success, frame = cam.read()
        frame = crop_image.get_crop_frame_from_frame(frame, image_id)
        integer_shapes = [int(hdiv * frame.shape[0]), int(wdiv * frame.shape[1])]
        a = [frame[:integer_shapes[0], :integer_shapes[1]],
             frame[:integer_shapes[0], integer_shapes[1]:],
             frame[integer_shapes[0]:, :integer_shapes[1]],
             frame[integer_shapes[0]:, integer_shapes[1]:]]
        pframe = frame = a[quater_id]
        isclosed |= not isOpen("background take")
        cv2.imshow("background take", frame)
        cv2.waitKey(1)
    cv2.imwrite(to_file, pframe)
    cv2.destroyWindow("background take")
    return pframe


def catch_frames(to_file: str, from_file: str = "../materials_part1/5.mkv", hdiv: float = 1, wdiv: float = 1,
                 image_id: int = 0) -> numpy.ndarray:
    data = list()
    for i in range(4):
        filename, file_extension = os.path.splitext(to_file)
        data.append(catch_frame(filename + "_tmp" + file_extension, from_file, hdiv, wdiv, image_id, i))

    total = numpy.concatenate((numpy.concatenate(
        (data[0], data[1]), axis=1), numpy.concatenate(
        (data[2], data[3]), axis=1)), axis=0)
    cv2.imwrite(to_file, total)
    cv2.imshow("total", total)
    cv2.waitKey(0)
    return total


catch_frames("../tmp/hehe.png", hdiv=0.2, wdiv=0.4)


array([[[ 50, 119,  95],
        [ 50, 119,  95],
        [ 50, 119,  95],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[ 49, 118,  94],
        [ 49, 118,  94],
        [ 49, 118,  94],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[ 47, 116,  92],
        [ 48, 117,  93],
        [ 50, 119,  95],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       ...,

       [[  2,   0,   0],
        [  2,   0,   0],
        [  2,   0,   0],
        ...,
        [  8,   0,   1],
        [  8,   0,   1],
        [  6,   0,   1]],

       [[  2,   0,   0],
        [  2,   0,   0],
        [  2,   0,   0],
        ...,
        [  8,   0,   1],
        [  8,   0,   1],
        [  6,   0,   1]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  2,   0,   0],
        ...,
        [  2,   0,   0],
        [  2,   0,   0],
        [  2,   0,   0]]

попытка поиска изображения мяча на картинке, проприетарная функция, требуется дальнейшая работа чтобы это заработало

In [3]:
import cv2
import numpy as np
from cv2 import xfeatures2d


# import sys

def readme():
    print(" Usage: python script.py <img1> <img2>")


def main():
    # if len(sys.argv) != 3:
    #     readme()
    #     return -1
    paths = ["C:\\Users\\shishkin_i\\Downloads\\Telegram Desktop\\ball.jpg",
             "F:\\PycharmProjects\\Camera-accuracy-studies\\tmp\\hehe_tmp.png"]
    img_object = cv2.imread(paths[0], cv2.IMREAD_GRAYSCALE)
    img_scene = cv2.imread(paths[1], cv2.IMREAD_GRAYSCALE)

    if img_object is None or img_scene is None:
        print(" --(!) Error reading images ")
        return -1

    #-- Step 1: Detect the keypoints using SURF Detector, then calculate the descriptors (feature vectors)
    minHessian = 400
    surf = xfeatures2d.SURF_create(hessianThreshold=minHessian)
    keypoints_object, descriptors_object = surf.detectAndCompute(img_object, None)
    keypoints_scene, descriptors_scene = surf.detectAndCompute(img_scene, None)

    #-- Step 2: Matching descriptor vectors using FLANN matcher
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.match(descriptors_object, descriptors_scene)

    matches = sorted(matches, key=lambda x: x.distance)

    #-- Quick calculation of max and min distances between keypoints
    min_dist = matches[0].distance
    max_dist = matches[-1].distance

    print("-- Max dist : {}".format(max_dist))
    print("-- Min dist : {}".format(min_dist))

    #-- Draw only "good" matches (i.e. whose distance is less than 3*min_dist )
    good_matches = [m for m in matches if m.distance < 3 * min_dist]

    img_matches = cv2.drawMatches(img_object, keypoints_object, img_scene, keypoints_scene, good_matches, None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    #-- Localize the object
    obj = np.float32([keypoints_object[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    scene = np.float32([keypoints_scene[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(obj, scene, cv2.RANSAC)

    #-- Get the corners from the image_1 (the object to be "detected")
    h, w = img_object.shape
    obj_corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    scene_corners = cv2.perspectiveTransform(obj_corners, H)

    #-- Draw lines between the corners (the mapped object in the scene - image_2 )
    img_scene_copy = img_matches.copy()
    scene_corners = np.int32(scene_corners + [w, 0])
    cv2.polylines(img_scene_copy, [scene_corners], True, (0, 255, 0), 4, cv2.LINE_AA)

    #-- Show detected matches
    cv2.imshow("Good Matches & Object detection", img_scene_copy)
    cv2.waitKey(0)


print("hehe")
if __name__ == "__main__":
    main()
    print("hehe")

hehe


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv_contrib\modules\xfeatures2d\src\surf.cpp:1028: error: (-213:The function/feature is not implemented) This algorithm is patented and is excluded in this configuration; Set OPENCV_ENABLE_NONFREE CMake option and rebuild the library in function 'cv::xfeatures2d::SURF::create'


попытка поиска изображения мяча на картинке, опенсорсная функция, все работает, но очень плохо. почему? непонятно...

In [36]:
import cv2
import numpy as np


def readme():
    print(" Usage: python script_name.py <img1> <img2>")


def main():
    paths = ["C:\\Users\\shishkin_i\\Downloads\\Telegram Desktop\\ball.jpg",
             "F:\\PycharmProjects\\Camera-accuracy-studies\\tmp\\hehe_tmp.png"]
    img_object = cv2.imread(paths[0], cv2.IMREAD_COLOR)
    img_scene = cv2.imread(paths[1], cv2.IMREAD_COLOR)

    if img_object is None or img_scene is None:
        print(" --(!) Error reading images ")
        return -1

    # Initiate ORB detector
    orb = cv2.ORB_create()

    # Find the keypoints and descriptors with ORB
    kp_object, des_object = orb.detectAndCompute(img_object, None)
    kp_scene, des_scene = orb.detectAndCompute(img_scene, None)

    # Create BFMatcher object
    bf = cv2.BFMatcher(cv2.NORM_L1, crossCheck=True)  #можно менять норму

    # Match descriptors
    matches = bf.match(des_object, des_scene)

    # Sort them in the order of their distance
    matches = sorted(matches, key=lambda x: x.distance)

    # Take the top 90% matches
    good_matches = matches[int(len(matches) * 0.):int(len(matches) * 0.9)]  # что вообще происходит?

    # Draw matches
    img_matches = cv2.drawMatches(img_object, kp_object, img_scene, kp_scene,
                                  good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    # Localize the object
    src_pts = np.float32([kp_object[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Find the Homography Matrix
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Get the corners from the object image
    h, w, _ = img_object.shape
    pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)

    # Project corners into the scene image
    dst = cv2.perspectiveTransform(pts, M)

    # Draw the projected rectangle on the scene image
    img_scene_with_rect = cv2.polylines(img_scene.copy(), [np.int32(dst)], True, (0, 255, 0), 4)

    # Display results
    cv2.imshow('Good Matches & Object detection', img_matches)
    cv2.imshow('Object detection', img_scene_with_rect)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    import sys

    main()

In [4]:

if __name__ == '__main__':
    i = 500
    path = '../materials_part1/5.mkv'
    cam = cv2.VideoCapture(path)
    cv2.namedWindow('test')
    width = cam.get(cv2.CAP_PROP_FRAME_WIDTH)  # float `width`
    height = cam.get(cv2.CAP_PROP_FRAME_HEIGHT)  # float `height`
    # previous = numpy.ndarray((int(height), int(width), 3), numpy.float64)
    previous = numpy.ndarray((359, 479, 3), numpy.float64)
    # print(width, height)
    previous_memorize_cntr = 100
    sum = 0
    pframe = 0
    isclosed = False
    # previous = numpy.concatenate(
    #     (cv2.imread("../tmp/v5_up_bacground.png"), cv2.imread("../tmp/v5_down2_bacground.png")), axis=0)
    previous = cv2.imread("../tmp/v5_bacground.png")
    # cv2.imwrite("../tmp/v5_bacground.png", previous)
    # cv2.imshow("test", previous)
    # while True:
    #     cv2.imshow("test", previous)
    #     cv2.waitKey(1)

    while not isclosed:
        isclosed |= not isOpen("test")
        # keyCode = cv2.waitKey(50)
        success, frame = cam.read()
        if success == False:
            cam.release()
            cam = cv2.VideoCapture(path)
            continue
        frame = crop_image.get_crop_frame_from_frame(frame, 1)
        # frame = frame[240:, :]
        pframe = frame
        # cv2.imwrite("../tmp/v5_down_bacground.png", frame)
        # break
        # if frame.shape != previous.shape:
        #     previous = numpy.ndarray(frame.shape, numpy.float64)
        frame_modified = frame.astype(numpy.float64)
        frame_modified = numpy.absolute(frame_modified - previous)
        # if previous_memorize_cntr > 0:
        #     previous = frame * 0.1 + previous * 0.9
        #     previous_memorize_cntr -= 1
        # if True:
        #     previous = (frame + previous * sum) / (sum + 1)
        #     sum += 1
        # print(frame.dtype)
        frame = frame_modified.astype(numpy.uint8)
        # break
        isclosed |= not isOpen("test")
        cv2.imshow("test", frame)
        cv2.waitKey(1)

    # cv2.imwrite("../tmp/v5_down2_bacground.png", pframe)

In [5]:


def nothing(x: int):
    pass


class memory_p_i_v1:
    def __init__(self):
        pass

    def init_settings(self, windowname: str):
        if self.setttings_set == False:
            return
        cv2.createTrackbar("lh", windowname, 60, 255, nothing)
        cv2.createTrackbar("ls", windowname, 28, 255, nothing)
        cv2.createTrackbar("lv", windowname, 57, 255, nothing)
        cv2.createTrackbar("hh", windowname, 168, 255, nothing)
        cv2.createTrackbar("hs", windowname, 255, 255, nothing)
        cv2.createTrackbar("hv", windowname, 255, 255, nothing)
        self.settings_set = True


def prepare_image_v1(background: numpy.ndarray, image: numpy.ndarray, windowname: str, settings: bool = False,
                     mem: memory_p_i_v1 = None) -> numpy.ndarray:
    if settings and mem != None:
        mem.init_settings(windowname)
    (lh, ls, lv, hh, hs, hv) = (60, 28, 57, 168, 255, 255)
    if settings and mem != None:
        lh = cv2.getTrackbarPos("lh", windowname)
        ls = cv2.getTrackbarPos("ls", windowname)
        lv = cv2.getTrackbarPos("lv", windowname)
        hh = cv2.getTrackbarPos("hh", windowname)
        hs = cv2.getTrackbarPos("hs", windowname)
        hv = cv2.getTrackbarPos("hv", windowname)
    frame_modified = image.astype(numpy.float64)
    frame_modified = numpy.maximum(frame_modified - background, 0)
    frame = frame_modified.astype(numpy.uint8)
    frame2 = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    frame2 = cv2.inRange(frame2, (lh, ls, lv), (hh, hs, hv))
    #smoothing
    w = numpy.asarray([[1, 1, 1],
                       [1, 1, 1],
                       [1, 1, 1]], dtype=numpy.uint8)
    frame2 = cv2.filter2D(frame2, -1, cv2.flip(w, -1), borderType=cv2.BORDER_CONSTANT)
    return frame2


class memory_c_i:
    def __init__(self):
        pass


def cluster_image(image: numpy.ndarray, remove_image: bool = False, mem: memory_c_i = None) -> tuple[
    numpy.ndarray, tuple[float, float]]:
# # Define criteria = ( type, max_iter = 10 , epsilon = 1.0 )
# criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
# # Set flags (Just to avoid line break in the code)
# flags = cv2.KMEANS_RANDOM_CENTERS
# # Apply KMeans
# compactness, labels, centers = cv2.kmeans(image, 2, None, criteria, 10, flags)

# Apply the Component analysis function
    (totalLabels, label_ids, values, centroid) = cv2.connectedComponentsWithStats(image, 4, cv2.CV_32S)
    if remove_image:
        image = image * 0
    pos = (-1, -1)
    for i in range(totalLabels):
        # print(values[i])
        if 250 < values[i][4] < 500:
            image = cv2.rectangle(image, (values[i][0], values[i][1]),
                                  (values[i][0] + values[i][2], values[i][1] + values[i][3]), (155, 155, 155), 2)
            pos = (values[i][0] + values[i][2] / 2, values[i][1] + values[i][1] / 2)
    return image, pos


In [6]:
path = '../materials_part1/5.mkv'
cam = cv2.VideoCapture(path)
cv2.namedWindow('test')
sum = 0
pframe = 0
isclosed = False
previous = cv2.imread("../tmp/v5_bacground.png")
prev_pos = None
path = list()
while not isclosed:
    isclosed |= not isOpen("test")
    if isclosed:
        break
    # keyCode = cv2.waitKey(50)
    success, frame = cam.read()
    if success == False:
        cam.release()
        cam = cv2.VideoCapture(path)
        continue
    frame = crop_image.get_crop_frame_from_frame(frame, 1)
    frame = prepare_image_v1(previous, frame, "test")
    frame, pos = cluster_image(frame)
    if prev_pos != None and pos != (-1, -1):
        prev_pos = ((prev_pos[0] + pos[0]) / 2, (prev_pos[1] + pos[1]) / 2)
    else:
        prev_pos = pos
    path.append(prev_pos)
    isclosed |= not isOpen("test")
    # cv2.imshow("test", frame)
    cv2.imshow("test", frame)
    cv2.waitKey(1)
print(path)

[(np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)), (np.float64(293.5), np.float64(364.5)),